In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import StratifiedShuffleSplit, StratifiedKFold, cross_val_score
import os

np.random.seed(42)
os.makedirs("../figures", exist_ok=True)

Upload the labels.csv and processed_counts.csv files to colab or your local workspace.

**Copied from Part 1:**
This data associates a cell barcode, such as "AAAGCCTGGCTAAC-1", to a certain cell type label, such as "CD14+ Monocyte". For each cell barcode, there are also log RNA seq counts of 765 different genes, such as HES4.

label.csv stores the association between a cell barcode and a cell type label.

processed_counts.csv stores the normalized log read counts for each cell, where each row represents a single cell, and each column represents a gene.

In [ ]:
labels_pd = pd.read_csv("../data/labels.csv")
counts_pd = pd.read_csv("../data/processed_counts.csv")

In [ ]:
labels_pd.index = labels_pd['index']
labels_pd.drop("index", axis=1, inplace=True)
counts_pd.index = counts_pd['Unnamed: 0']
counts_pd.drop("Unnamed: 0", axis=1, inplace=True)

df = counts_pd.merge(labels_pd, left_index=True, right_index=True).dropna()
df

One-hot encode the cell-type.

Shuffle your data. Make sure your labels and the counts are shuffled together.

Split into train and test sets (80:20 split)

In [ ]:
categories = df['bulk_labels'].unique()
print(categories)

# one-hot encoding
y_onehot = np.zeros((len(df), len(categories)))
for i, ct in enumerate(df['bulk_labels']):
    y_onehot[i, np.where(categories == ct)[0]] = 1

le = LabelEncoder()
y = le.fit_transform(df['bulk_labels'].values)
X = df.drop('bulk_labels', axis=1).values.astype(np.float32)

# stratified split preserves class proportions — important for 10-class imbalanced data
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(sss.split(X, y))

X_train, X_test   = X[train_idx], X[test_idx]
y_train, y_test   = y[train_idx], y[test_idx]
oh_train, oh_test = y_onehot[train_idx], y_onehot[test_idx]

In [ ]:
# Visualize the One-hot encoded Prediction Labels
plt.figure(figsize=(9, 3), dpi=150)
plt.imshow(oh_train[:50])
plt.xlabel("cell type"); plt.ylabel("cell index")
plt.tight_layout()
plt.savefig('../figures/onehot_labels.png', dpi=150, bbox_inches='tight')
plt.show()

Apply classification algorithms to the training data, tune on validation data (if present), and evaluate on test data.

You can also apply classification downstream of last week's autoencoder latent space representation.

In [ ]:
# balanced weights matter here — smallest class has 8 samples
rf = RandomForestClassifier(
    n_estimators=300,
    class_weight='balanced',
    n_jobs=-1,
    random_state=42
)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

acc = accuracy_score(y_test, y_pred)
print(f"accuracy: {acc:.4f}\n")
print(classification_report(y_test, y_pred, target_names=le.classes_, zero_division=0))

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(
    RandomForestClassifier(n_estimators=300, class_weight='balanced', n_jobs=-1, random_state=42),
    X, y, cv=cv, scoring='accuracy'
)
print(f"5-fold CV: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
print(f"Per-fold:  {[f'{s:.3f}' for s in cv_scores]}")

In [ ]:
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(cm, annot=True, fmt='d',
            xticklabels=le.classes_, yticklabels=le.classes_,
            cmap='Blues', ax=ax, annot_kws={"size": 9})
ax.set_xlabel('predicted'); ax.set_ylabel('true')
ax.set_title(f'Random Forest — test accuracy {acc:.3f}')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.savefig('../figures/part2_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
report = classification_report(y_test, y_pred,
                               target_names=le.classes_, output_dict=True, zero_division=0)
f1s  = [report[ct]['f1-score'] for ct in le.classes_]
supp = [report[ct]['support']  for ct in le.classes_]

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(le.classes_, f1s, color='steelblue')
for bar, s in zip(bars, supp):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'n={int(s)}', ha='center', va='bottom', fontsize=8)
ax.axhline(acc, color='crimson', linestyle='--', alpha=0.7, label=f'accuracy={acc:.3f}')
ax.set_ylabel('F1'); ax.set_ylim(0, 1.2)
ax.set_title('Per-class F1 — Random Forest')
ax.legend()
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.tight_layout()
plt.savefig('../figures/part2_f1_scores.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
importances = rf.feature_importances_
gene_names = df.drop('bulk_labels', axis=1).columns
top_idx = np.argsort(importances)[::-1][:20]

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(range(20), importances[top_idx], color='steelblue')
ax.set_xticks(range(20))
ax.set_xticklabels(gene_names[top_idx], rotation=45, ha='right', fontsize=9)
ax.set_ylabel('Feature importance')
ax.set_title('Top 20 genes by Random Forest importance')
plt.tight_layout()
plt.savefig('../figures/part2_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## Classifier

A Random Forest with 300 trees and balanced class weights was trained on the 765-gene expression features using a stratified 80/20 split, achieving 85.0% test accuracy (5-fold CV: 81.6% ± 1.5%). Stratified splitting was important here — with a 30:1 class imbalance, a random split can easily leave the rarest classes barely represented in the test set.

F1 scores give a more honest picture than accuracy alone. The model does well on the larger classes: CD19+ B and Dendritic cells both have F1 above 0.90, and CD14+ Monocyte is close at 0.89. Smaller classes are harder — CD8+ Cytotoxic T drops to 0.62. Two classes, CD4+/CD45RA+/CD25- Naive T and CD4+/CD45RO+ Memory, have F1=0, but those have only 1 and 4 test samples respectively. There's not enough data there to say much.

The top genes by feature importance include CD79A, GNLY, NKG7, and CD3D — known markers for B cells, NK/cytotoxic T cells, and T cells — alongside monocyte markers like LYZ and CST3. The model is picking up on real biological signal rather than noise.